# Credit Card Customer Churn & Risk Segmentation Dashboard

### Question: Which credit card customer segments are most likely to churn, and what account and usage patterns can predict that risk?

### Stakeholder Framing: A bank's customer retention team requires a queryable tool that ranks currently active customers by churn risk so that retention offers can be targeted at the right customers before they leave.

### Flask app for dashboard

In [28]:
%%writefile app/app.py
from pathlib import Path

from flask import Flask, jsonify, render_template
from sqlalchemy import create_engine
import pandas as pd

APP_DIR = Path(__file__).resolve().parent
DB_PATH = APP_DIR.parent / "db" / "churn.db"

app = Flask(__name__)
engine = create_engine(f"sqlite:///{DB_PATH}")


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/api/top-risk-customers")
def top_risk_customers():
    df = pd.read_sql(
        "SELECT * FROM customer_scores WHERE attrition_flag = 'Existing Customer' "
        "ORDER BY churn_risk_score DESC LIMIT 25", engine)
    return jsonify(df.to_dict(orient="records"))


@app.route("/api/summary-stats")
def summary_stats():
    df = pd.read_sql(
        "SELECT attrition_flag, COUNT(*) as customer_count, "
        "AVG(total_trans_ct) as avg_trans_ct, AVG(avg_utilization_ratio) as avg_utilization "
        "FROM customer_scores GROUP BY attrition_flag", engine)
    return jsonify(df.to_dict(orient="records"))


if __name__ == "__main__":
    app.run(debug=True)

Overwriting app/app.py


### HTML index for dashboard

In [29]:
%%writefile app/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Customer Churn Risk Dashboard</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
    <script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js"></script>
</head>
<body>
    <header>
        <h1>Customer Churn Risk Dashboard</h1>
        <p>Credit card customer retention — active customers ranked by churn risk</p>
    </header>

    <section class="summary-cards" id="summaryCards"></section>

    <section class="chart-section">
        <h2>Top 15 At-Risk Active Customers</h2>
        <canvas id="riskChart" height="100"></canvas>
    </section>

    <section class="table-section">
        <h2>At-Risk Customer Detail</h2>
        <table id="riskTable">
            <thead>
                <tr>
                    <th>Client #</th>
                    <th>Age</th>
                    <th>Income Category</th>
                    <th>Card Category</th>
                    <th>Trans. Count</th>
                    <th>Utilization</th>
                    <th>Risk Score</th>
                </tr>
            </thead>
            <tbody></tbody>
        </table>
    </section>

    <script src="{{ url_for('static', filename='dashboard.js') }}"></script>
</body>
</html>

Overwriting app/templates/index.html


### CSS for dashboard

In [30]:
%%writefile app/static/style.css
* { box-sizing: border-box; }

body {
    font-family: -apple-system, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
    background: #f7f8fa;
    color: #222;
    margin: 0;
    padding: 0 24px 48px;
}

header {
    padding: 32px 0 16px;
    border-bottom: 1px solid #e2e4e8;
    margin-bottom: 24px;
}

header h1 {
    margin: 0 0 4px;
    font-size: 1.6rem;
}

header p {
    margin: 0;
    color: #666;
}

.summary-cards {
    display: flex;
    gap: 16px;
    margin-bottom: 32px;
    flex-wrap: wrap;
}

.card {
    background: white;
    border: 1px solid #e2e4e8;
    border-radius: 8px;
    padding: 16px 20px;
    min-width: 180px;
}

.card .label {
    font-size: 0.8rem;
    color: #777;
    text-transform: uppercase;
    letter-spacing: 0.03em;
}

.card .value {
    font-size: 1.5rem;
    font-weight: 600;
    margin-top: 4px;
    color: #C44E52;
}

.chart-section, .table-section {
    background: white;
    border: 1px solid #e2e4e8;
    border-radius: 8px;
    padding: 20px 24px;
    margin-bottom: 24px;
}

table {
    width: 100%;
    border-collapse: collapse;
    font-size: 0.9rem;
}

th, td {
    text-align: left;
    padding: 8px 10px;
    border-bottom: 1px solid #eee;
}

th {
    cursor: pointer;
    color: #444;
    user-select: none;
}

th:hover {
    color: #C44E52;
}

Overwriting app/static/style.css


### javascript for dashboard

In [31]:
%%writefile app/static/dashboard.js
async function fetchJSON(url) {
    const res = await fetch(url);
    return res.json();
}

function renderSummaryCards(summary) {
    const container = document.getElementById("summaryCards");
    const existing = summary.find(d => d.attrition_flag === "Existing Customer") || {};
    const attrited = summary.find(d => d.attrition_flag === "Attrited Customer") || {};
    const totalCustomers = (existing.customer_count || 0) + (attrited.customer_count || 0);
    const attritionRate = totalCustomers ? ((attrited.customer_count || 0) / totalCustomers * 100).toFixed(1) : "0";

    const cards = [
        { label: "Total Customers", value: totalCustomers },
        { label: "Attrition Rate", value: `${attritionRate}%` },
        { label: "Avg Trans. Count (Existing)", value: (existing.avg_trans_ct || 0).toFixed(1) },
        { label: "Avg Trans. Count (Attrited)", value: (attrited.avg_trans_ct || 0).toFixed(1) },
    ];

    container.innerHTML = cards.map(c => `
        <div class="card">
            <div class="label">${c.label}</div>
            <div class="value">${c.value}</div>
        </div>
    `).join("");
}

function renderRiskChart(customers) {
    const ctx = document.getElementById("riskChart");
    new Chart(ctx, {
        type: "bar",
        data: {
            labels: customers.map(c => c.client_num),
            datasets: [{
                label: "Churn Risk Score",
                data: customers.map(c => c.churn_risk_score),
                backgroundColor: "#C44E52"
            }]
        },
        options: {
            responsive: true,
            plugins: { legend: { display: false } },
            scales: { x: { ticks: { autoSkip: false, maxRotation: 60 } } }
        }
    });
}

function renderTable(customers) {
    const tbody = document.querySelector("#riskTable tbody");
    tbody.innerHTML = customers.map(c => `
        <tr>
            <td>${c.client_num}</td>
            <td>${c.customer_age}</td>
            <td>${c.income_category}</td>
            <td>${c.card_category}</td>
            <td>${c.total_trans_ct}</td>
            <td>${(c.avg_utilization_ratio ?? 0).toFixed(2)}</td>
            <td>${c.churn_risk_score.toFixed(3)}</td>
        </tr>
    `).join("");
}

(async function init() {
    const [summary, topRisk] = await Promise.all([
        fetchJSON("/api/summary-stats"),
        fetchJSON("/api/top-risk-customers")
    ]);
    renderSummaryCards(summary);
    renderRiskChart(topRisk.slice(0, 15));
    renderTable(topRisk);
})();

Overwriting app/static/dashboard.js


## To Run the Web Dashboard

1. Open a terminal in this project's root folder.
2. Run:
   ```bash
   python app/app.py
   ```
3. Open **http://127.0.0.1:5000** in your browser.

You can keep re-running this notebook to refresh `db/churn.db` with new data or a revised risk score — the Flask app will pick up the changes on its next request since it reads the SQL tables live, no restart needed for data changes (only for code changes in `app.py` itself).